# 00 · Configuración del entorno

Funciona en **Google Colab** y en **escritorio** (VS Code / Jupyter). La configuración es igual para todo el equipo: no hay que editar nada.

**Regla del equipo:** el código se escribe en VS Code y se sube a GitHub desde la computadora. Colab solo baja el código y entrena.

Requisitos (una sola vez por persona):
1. En Google Drive: *Compartidos conmigo* → clic derecho en `ProyectoIA_ComandosDeVoz` → *Organizar → Agregar acceso directo* → *Mi unidad*.
2. En Colab: icono de llave 🔑 (*Secrets*) → crear `WANDB_API_KEY` con tu clave de https://wandb.ai/authorize y activar *Acceso al notebook*.
3. Solo en escritorio, si Drive no está en `G:\Mi unidad` ni en `G:\My Drive`: definir la variable de entorno `DRIVE_ROOT` con tu ruta.

## 1. Configuración del proyecto (igual para todas)

In [ ]:
REPO_URL      = "https://github.com/JennyKing18/IA.git"
DRIVE_FOLDER  = "ProyectoIA_ComandosDeVoz"
WANDB_PROJECT = "comandos-de-voz"
WANDB_ENTITY  = None   # nombre del equipo en W&B, o None

## 2. Detectar entorno y montar Drive

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def find_drive_root():
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive")
    if os.environ.get("DRIVE_ROOT"):
        return Path(os.environ["DRIVE_ROOT"])
    candidates = [Path(r"G:\Mi unidad"), Path(r"G:\My Drive"), *Path.home().glob("Library/CloudStorage/GoogleDrive-*/*")]
    for c in candidates:
        if (c / DRIVE_FOLDER).exists():
            return c
    raise FileNotFoundError("No encontré Google Drive. Define la variable de entorno DRIVE_ROOT con la ruta de 'Mi unidad'.")

DRIVE_ROOT    = find_drive_root()
PROJECT_DRIVE = DRIVE_ROOT / DRIVE_FOLDER
DATA_DIR      = PROJECT_DRIVE / "speech_commands_v0.02"
CKPT_DIR      = PROJECT_DRIVE / "checkpoints"
EXPORT_DIR    = PROJECT_DRIVE / "exports"
for d in (CKPT_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Colab:", IN_COLAB)
print("Carpeta del proyecto:", PROJECT_DRIVE, "| existe:", PROJECT_DRIVE.exists())

## 3. Obtener el código

In [ ]:
def run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r.returncode

if IN_COLAB:
    REPO_DIR = Path("/content") / Path(REPO_URL).stem
    if not REPO_DIR.exists():
        run(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        run("git pull --ff-only", cwd=REPO_DIR)
else:
    # En escritorio el notebook se abre desde el repo clonado (carpeta notebooks/).
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

## 4. Instalar librerías

In [ ]:
if IN_COLAB:
    run("pip install -q -r requirements.txt")
else:
    print("En escritorio: instala PyTorch desde pytorch.org y luego `pip install -r requirements.txt` en tu entorno.")

## 5. Semillas y dispositivo

In [ ]:
import torch
from src.config import COMMANDS, SEED, set_seed

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| dispositivo:", DEVICE)

## 6. Verificar el dataset

In [ ]:
problemas = []
if not DATA_DIR.exists():
    problemas.append(f"No existe {DATA_DIR}")
else:
    for c in COMMANDS:
        n = len(list((DATA_DIR / c).glob("*.wav"))) if (DATA_DIR / c).exists() else 0
        print(f"{c:>6}: {n} archivos")
        if n == 0:
            problemas.append(f"Falta la carpeta o está vacía: {c}")
    for lst in ("validation_list.txt", "testing_list.txt"):
        p = DATA_DIR / lst
        if p.exists():
            print(f"{lst}: {sum(1 for _ in open(p))} líneas")
        else:
            problemas.append(f"Falta {lst}")

print("\nOK" if not problemas else "\nPROBLEMAS:\n- " + "\n- ".join(problemas))

## 7. Iniciar sesión en Weights & Biases

In [ ]:
import wandb

key = None
if IN_COLAB:
    from google.colab import userdata
    try:
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
wandb.login(key=key) if key else wandb.login()   # en escritorio pide la clave la primera vez y la recuerda

## 8. Prueba de conexión con W&B

Sube un run corto con métricas inventadas (no entrena nada). Abre el enlace que imprime y revisa que aparezcan las gráficas y la matriz de confusión. Después puedes borrar el run en W&B.

In [ ]:
from src.wandb_check import run_check
run_check(entity=WANDB_ENTITY, project=WANDB_PROJECT)